# Reproducing Figure 3A–C: sequential associative-memory learning

**Paper:** T. Kurikawa and K. Kaneko, *Fast Neural Noise Stabilizes Associative Memory against Slow Synaptic Noise* — Figure 3A, 3B, and 3C.

This stand-alone notebook was extracted from the authors' original final-paper notebook. It **recreates Fig. 3A–C from the numerical neural–synaptic simulations only**: the AMF analysis, first-memory calibration, and auxiliary figures are not necessary for these three panels and have been removed. The original simulation functions are preserved without changing their numerical update rules.

**To reproduce the publication results:** set `RUN_MODE = "paper"` below and run all cells. Outputs are saved under a versioned run directory. The paper configuration is computationally substantial (**80 noise conditions, 880 independently trained networks, 20 patterns per network**); a CUDA GPU is strongly recommended. For a fast check that the pipeline runs, change `RUN_MODE` to `"smoke"` (these outputs are **not** paper figures).

**Generated files:** `Fig3A.pdf/png`, `Fig3B.pdf/png`, `Fig3C.pdf/png`, and `Fig3_ABC.pdf/png`, plus `fig3_recall_trials.csv`, `fig3_network_memory.csv`, `fig3_network_capacity.csv`, `fig3A_summary.csv`, and `fig3BC_summary.csv`. Results are checkpointed after each complete network, so interrupted calculations can be resumed.

**Environment:** Python 3.10+, NumPy, pandas, matplotlib; PyTorch with CUDA is optional (recommended). NumPy is used automatically on systems without CUDA. Torch and NumPy have different random streams and floating-point trajectories; the manuscript's simulated *trends* should be reproducible, but bit-for-bit equality across backends is not expected. No external input data or Google Drive files are required.

## 1. Configuration and manuscript protocol

The implemented equations are the original neural–synaptic SDEs (manuscript Eqs. 1–2). Learning is performed sequentially for 20 independent binary input–target pairs at `N=200`, `beta=10`, `alpha=0.001`, `tauJ=10` with Euler–Maruyama `dt=0.01`. Each episode ends at the **first post-update time** the instantaneous overlap reaches `0.99` (not a convergence/steady-state threshold). Neural activity, weights and noise streams continue across episodes; `J_ii` is fixed to zero.

After the final episode, the synaptic matrix is frozen and both noises are removed. For each of five independently initialized recalls **per trained network and memory**, the overlap is time averaged over `50 <= t <= 100`. The network-level capacity is the mean overlap over its 20 memories and five recalls. The standard error of the mean (SEM) is calculated **across trained networks**, never across the five recalls as if they were independent networks.

- **Fig. 3A:** `Ds=0.01`, `Dn=0.01, 0.25, 0.5, 1.0`; **30 independently trained networks per condition**; memory index from oldest (0) to newest (19).
- **Fig. 3B:** average memory overlap versus `Dn`, at `Ds=0.01` and `Ds=1.0`.
- **Fig. 3C:** average memory overlap versus `Ds`, at `Dn=0.01` and `Dn=1.0`.

For B/C each sweep contains `0.01` plus `0.05, 0.10, ..., 1.00`. Overlapping A conditions use their 30-network cohorts; all other conditions use 10 networks. Thus `80` unique noise pairs and `880` trained networks are needed. The initial maps, initial weights, neural activities and recall seeds are paired by **network ID** across conditions as in the original notebook.

*To use Google Colab + Drive*, set `USE_GOOGLE_DRIVE=True`. Otherwise the notebook writes to a local `fig3_reproduction_outputs/` directory. The default `RUN_MODE="paper"` preserves the source model and its sample sizes.

In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import matplotlib.pyplot as plt

# === User options ============================================================
RUN_MODE = "paper"       # "paper": reproduce Fig. 3A–C; "smoke": small pipeline test only
USE_GOOGLE_DRIVE = False  # True in Colab for persistent checkpointing
RUN_TRAINING = True      # False: only aggregate / redraw completed cached simulations
DEVICE = "auto"         # "auto" (CUDA if available, otherwise NumPy), "numpy", "cuda"
OUTPUT_BASE = Path("fig3_reproduction_outputs")
SHOW_FIGURES = True

# These must remain unchanged for manuscript figure reproduction.
if RUN_MODE == "paper":
    SIM_SETTINGS = dict(
        N=200, P=20, beta=10.0, alpha=0.001, tauJ=10.0,
        dt=0.01, target=0.99, max_train_time=32000.0, seed=1729,
        device=DEVICE, dtype="float32", torch_chunk_size=64,
        train_init="uniform", recall_init="uniform", reset_between_patterns=False,
        check_interval=1, blas_threads=1, n_recall=5,
        recall_time=100.0, recall_burn=50.0,
    )
    BASIC_GRID = [0.01] + [round(i * 0.05, 8) for i in range(1, 21)]
    N_NETWORKS = 10
    N_FIG3A_NETWORKS = 30
    BATCH_SIZE = 5
elif RUN_MODE == "smoke":
    # Entire pipeline test. Results do NOT represent the paper experiment.
    SIM_SETTINGS = dict(
        N=24, P=3, beta=10.0, alpha=0.001, tauJ=1.0,
        dt=0.05, target=0.10, max_train_time=60.0, seed=1729,
        device="numpy", dtype="float32", torch_chunk_size=8,
        train_init="uniform", recall_init="uniform", reset_between_patterns=False,
        check_interval=1, blas_threads=1, n_recall=2,
        recall_time=2.0, recall_burn=1.0,
    )
    BASIC_GRID = [0.01, 0.5, 1.0]
    N_NETWORKS = 2
    N_FIG3A_NETWORKS = 3
    BATCH_SIZE = 3
else:
    raise ValueError("RUN_MODE must be 'paper' or 'smoke'.")

FIG3A_DN = [0.01, 0.25, 0.5, 1.0]
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/fig3_reproduction_outputs")

print(f"Run mode: {RUN_MODE}; device: {SIM_SETTINGS['device']}; "
      f"N={SIM_SETTINGS['N']}, patterns={SIM_SETTINGS['P']}")


## 2. Simulation implementation (preserved from the original notebook)

The following four code cells contain the same random-stream allocation, first-passage criterion, Euler–Maruyama / Euler integration, and CUDA/NumPy backends as the authors' original notebook. They are separated into smaller cells for readability.

### Simulation configuration, RNG, and state initialization

In [ ]:
"""Full neural-synaptic simulation for the dual-noise, 20-pattern experiment.

This module implements the physical-time SDEs in Eqs. (1)-(2), with an
instantaneous first-passage learning criterion.  All computational choices
absent from the manuscript are explicit in DEFAULT_SIM_CONFIG.

Only NumPy is required.  PyTorch is imported only for the optional CUDA
backend.  A checkpoint callback accepts one event dictionary; its
``partial_result`` member is a live result dictionary, so a callback that
retains a checkpoint must copy arrays or serialize them before returning.
"""

from contextlib import nullcontext
import math
import time
from typing import Any, Callable, Mapping, Sequence

import numpy as np


DEFAULT_SIM_CONFIG = {
    "N": 200,
    "P": 20,
    "beta": 10.0,
    "alpha": 0.001,
    "tauJ": 10.0,
    "dt": 0.01,
    "target": 0.99,
    "max_train_time": 32000.0,
    "seed": 1729,
    "n_recall": 5,
    "recall_time": 100.0,
    "recall_burn": 50.0,
    "device": "auto",
    "dtype": "float32",
    "train_init": "uniform",
    "recall_init": "uniform",
    "reset_between_patterns": False,
    "check_interval": 1,
    "blas_threads": 1,
    "torch_chunk_size": 64,
}


class TrainingFailure(RuntimeError):
    """Training did not complete; ``partial_result`` preserves its state."""

    def __init__(self, message: str, partial_result: dict[str, Any]):
        super().__init__(message)
        self.partial_result = partial_result


def _sim_config(cfg: Mapping[str, Any]) -> dict[str, Any]:
    out = dict(DEFAULT_SIM_CONFIG)
    out.update(dict(cfg))
    for key in ("N", "P", "n_recall", "seed", "check_interval", "blas_threads", "torch_chunk_size"):
        value = out[key]
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)):
            raise ValueError(f"{key} must be an integer, got {value!r}.")
        out[key] = int(value)
    if out["N"] < 2 or out["P"] < 1 or out["n_recall"] < 1:
        raise ValueError("Require N >= 2, P >= 1 and n_recall >= 1.")
    if out["seed"] < 0 or out["blas_threads"] < 1 or out["torch_chunk_size"] < 1:
        raise ValueError("Require seed >= 0, blas_threads >= 1 and torch_chunk_size >= 1.")
    if out["check_interval"] != 1:
        raise ValueError("Instantaneous first passage requires check_interval=1.")
    for key in ("beta", "alpha", "tauJ", "dt", "target", "max_train_time",
                "recall_time", "recall_burn"):
        out[key] = float(out[key])
        if not np.isfinite(out[key]):
            raise ValueError(f"{key} must be finite.")
    if any(out[key] <= 0 for key in ("beta", "tauJ", "dt", "max_train_time", "recall_time")):
        raise ValueError("beta, tauJ, dt and integration durations must be positive.")
    if out["alpha"] < 0 or not -1 < out["target"] <= 1:
        raise ValueError("Require alpha >= 0 and -1 < target <= 1.")
    if not 0 <= out["recall_burn"] < out["recall_time"]:
        raise ValueError("Require 0 <= recall_burn < recall_time.")
    for key in ("max_train_time", "recall_time", "recall_burn"):
        ratio = out[key] / out["dt"]
        if not np.isclose(ratio, round(ratio), rtol=0, atol=1e-8 * max(1, abs(ratio))):
            raise ValueError(f"{key}/dt must be an integer; got {ratio}.")
    if out["dtype"] not in ("float32", "float64"):
        raise ValueError("dtype must be 'float32' or 'float64'.")
    for key in ("train_init", "recall_init"):
        if out[key] not in ("uniform", "normal", "zero"):
            raise ValueError(f"{key} must be 'uniform', 'normal' or 'zero'.")
    if not isinstance(out["reset_between_patterns"], (bool, np.bool_)):
        raise ValueError("reset_between_patterns must be a boolean.")
    out["reset_between_patterns"] = bool(out["reset_between_patterns"])
    return out


def _rng(seed: int, network_id: int, stream: int) -> np.random.Generator:
    # Stable seeds do not depend on condition ordering or batch composition.
    return np.random.default_rng(np.random.SeedSequence([seed, network_id, stream]))


def _torch_seed(seed: int, network_id: int, stream: int, pattern: int) -> int:
    words = np.random.SeedSequence([seed, network_id, stream, pattern]).generate_state(2)
    return int((int(words[0]) | (int(words[1]) << 32)) % (2**63 - 1))


def _initial_x(rng: np.random.Generator, shape: tuple[int, ...],
               mode: str, dtype: np.dtype) -> np.ndarray:
    if mode == "zero":
        return np.zeros(shape, dtype=dtype)
    if mode == "normal":
        return rng.standard_normal(shape, dtype=dtype)
    return rng.uniform(-1.0, 1.0, size=shape).astype(dtype)


def _notify(callback: Callable | None, result: dict[str, Any], event: str,
            **details: Any) -> None:
    if callback is not None:
        callback({"event": event, "Dn": result["Dn"], "Ds": result["Ds"],
                  "partial_result": result, **details})


def _make_result(cfg: dict[str, Any], Dn: float, Ds: float,
                 net_ids: np.ndarray, backend: str) -> dict[str, Any]:
    B, N, P = len(net_ids), cfg["N"], cfg["P"]
    dtype = np.dtype(cfg["dtype"])
    result = {
        "Dn": float(Dn), "Ds": float(Ds), "config": dict(cfg),
        "backend": backend, "net_ids": net_ids.copy(),
        "eta": np.empty((B, P, N), dtype=dtype),
        "xi": np.empty((B, P, N), dtype=dtype),
        "q": np.empty((B, P, N), dtype=dtype),
        "J_initial": np.empty((B, N, N), dtype=dtype),
        "J_final": np.empty((B, N, N), dtype=dtype),
        "initial_x": np.empty((B, N), dtype=dtype),
        "final_x": np.empty((B, N), dtype=dtype),
        "s_boundary": np.full((B, P, P, N), np.nan, dtype=dtype),
        "learn_steps": np.full((B, P), -1, dtype=np.int64),
        "learn_overlap": np.full((B, P), np.nan, dtype=np.float64),
        "recall": np.full((B, P, cfg["n_recall"]), np.nan, dtype=np.float64),
        "completed_patterns": np.zeros(B, dtype=np.int64),
        "training_complete": np.zeros(B, dtype=bool),
        "recall_complete": np.zeros(B, dtype=bool),
        "wall_seconds": 0.0,
        "protocol": {
            "time": "physical neural time; synaptic drift and amplitude divided by tauJ",
            "switch": "first post-update or initial overlap >= target; checked every dt",
            "training_integrator": "simultaneous Euler-Maruyama using pre-step x and J",
            "recall_integrator": "Euler, frozen J, no neural or synaptic noise",
            "recall_average": "uniform grid samples with recall_burn <= t <= recall_time",
            "noise_streams": "per-network independent map, weight, initial, neural, synaptic and recall streams; Torch training noise additionally separated by memory",
            "condition_pairing": "same initial maps/J and stream seeds for each network ID across conditions",
            "backend_reproducibility": "backend-specific floating-point and RNG trajectories; batch order does not change assigned streams",
            "q_definition": "xi * eta",
        },
    }
    for b, net_id in enumerate(net_ids):
        maps_rng = _rng(cfg["seed"], int(net_id), 0)
        result["eta"][b] = (2 * maps_rng.integers(0, 2, (P, N)) - 1).astype(dtype)
        result["xi"][b] = (2 * maps_rng.integers(0, 2, (P, N)) - 1).astype(dtype)
        J = _rng(cfg["seed"], int(net_id), 1).standard_normal((N, N), dtype=dtype)
        J /= np.sqrt(N)
        np.fill_diagonal(J, 0.0)
        result["J_initial"][b] = J
        result["J_final"][b] = J
        x = _initial_x(_rng(cfg["seed"], int(net_id), 2), (N,), cfg["train_init"], dtype)
        result["initial_x"][b] = x
        result["final_x"][b] = x
    result["q"][:] = result["xi"] * result["eta"]
    return result


def _failure(result: dict[str, Any], callback: Callable | None, *,
             message: str, network_id: int, pattern_index: int,
             steps: int, overlap: float) -> None:
    result["failure"] = {"message": message, "network_id": int(network_id),
                         "pattern_index": int(pattern_index), "steps": int(steps),
                         "overlap": float(overlap)}
    _notify(callback, result, "training_failed", **result["failure"])
    raise TrainingFailure(message, result)


def _record_boundary(result: dict[str, Any], callback: Callable | None,
                     b: int, mu: int, steps: int, overlap: float,
                     J: np.ndarray, x: np.ndarray) -> None:
    result["J_final"][b] = J
    result["final_x"][b] = x
    if not np.isfinite(J).all() or not np.isfinite(x).all() or not np.isfinite(overlap):
        _failure(result, callback, message=(
            f"Nonfinite state at a learning boundary: network={result['net_ids'][b]}, "
            f"pattern={mu}, step={steps}."), network_id=int(result["net_ids"][b]),
            pattern_index=mu, steps=steps, overlap=overlap)
    result["s_boundary"][b, mu] = result["xi"][b] * (
        result["xi"][b] @ J.T + result["eta"][b])
    result["learn_steps"][b, mu] = steps
    result["learn_overlap"][b, mu] = overlap
    result["completed_patterns"][b] = mu + 1
    _notify(callback, result, "pattern_learned", network_id=int(result["net_ids"][b]),
            network_index=b, pattern_index=mu, steps=int(steps), overlap=float(overlap))




### NumPy training and deterministic recall

In [ ]:
def _numpy_recall(result: dict[str, Any], b: int) -> np.ndarray:
    cfg = result["config"]
    N, P, R = cfg["N"], cfg["P"], cfg["n_recall"]
    dtype = np.dtype(cfg["dtype"])
    x = _initial_x(_rng(cfg["seed"], int(result["net_ids"][b]), 5),
                   (P, R, N), cfg["recall_init"], dtype).reshape(P * R, N)
    eta = np.repeat(result["eta"][b], R, axis=0)
    xi = np.repeat(result["xi"][b], R, axis=0)
    J = result["J_final"][b]
    n_steps = int(round(cfg["recall_time"] / cfg["dt"]))
    burn_step = int(round(cfg["recall_burn"] / cfg["dt"]))
    overlap_sum = np.zeros(P * R, dtype=np.float64)
    count = 0
    if burn_step == 0:
        overlap_sum += np.einsum("ij,ij->i", x, xi) / N
        count += 1
    for step in range(1, n_steps + 1):
        x += cfg["dt"] * (-x + np.tanh(cfg["beta"] * (x @ J.T + eta)))
        if step >= burn_step:
            overlap_sum += np.einsum("ij,ij->i", x, xi) / N
            count += 1
    if not np.isfinite(x).all() or count < 1:
        raise FloatingPointError("Recall integration produced nonfinite values or no samples.")
    return (overlap_sum / count).reshape(P, R)


def _train_numpy(result: dict[str, Any], callback: Callable | None) -> None:
    cfg, Dn, Ds = result["config"], result["Dn"], result["Ds"]
    N, P = cfg["N"], cfg["P"]
    dtype = np.dtype(cfg["dtype"])
    dt, tauJ = cfg["dt"], cfg["tauJ"]
    max_steps = int(round(cfg["max_train_time"] / dt))
    noise_x = np.sqrt(2 * Dn * dt)
    noise_J = np.sqrt(2 * Ds * dt / N) / tauJ
    for b, net_id in enumerate(result["net_ids"]):
        J, x = result["J_initial"][b].copy(), result["initial_x"][b].copy()
        neural_rng = _rng(cfg["seed"], int(net_id), 3)
        synaptic_rng = _rng(cfg["seed"], int(net_id), 4)
        reset_rng = _rng(cfg["seed"], int(net_id), 6)
        nx, nw = np.empty(N, dtype=dtype), np.empty((N, N), dtype=dtype)
        outer = np.empty((N, N), dtype=dtype)
        for mu in range(P):
            if mu and cfg["reset_between_patterns"]:
                x = _initial_x(reset_rng, (N,), cfg["train_init"], dtype)
            xi, eta = result["xi"][b, mu], result["eta"][b, mu]
            overlap = float(np.dot(xi, x) / N)
            steps = 0
            while overlap < cfg["target"] and steps < max_steps:
                # Both increments depend on x and J BEFORE this time step.
                row_norm = np.einsum("ij,ij->i", J, J)
                x_new = x + dt * (-x + np.tanh(cfg["beta"] * (J @ x + eta)))
                if Dn:
                    neural_rng.standard_normal(out=nx, dtype=dtype)
                    x_new += noise_x * nx
                np.multiply((xi - x)[:, None], x[None, :], out=outer)
                J *= (1 + (dt / tauJ) * cfg["alpha"] * (1 - row_norm))[:, None]
                J += (dt / (tauJ * N)) * outer
                if Ds:
                    synaptic_rng.standard_normal(out=nw, dtype=dtype)
                    J += noise_J * nw
                np.fill_diagonal(J, 0.0)
                x = x_new
                steps += 1
                overlap = float(np.dot(xi, x) / N)
                if not np.isfinite(overlap) or not np.isfinite(row_norm).all():
                    result["J_final"][b], result["final_x"][b] = J, x
                    _failure(result, callback, message=(
                        f"Nonfinite state at Dn={Dn}, Ds={Ds}, network={net_id}, "
                        f"pattern={mu}, t={steps * dt:g}. Use a smaller explicit dt."),
                        network_id=int(net_id), pattern_index=mu, steps=steps, overlap=overlap)
            if not overlap >= cfg["target"]:
                result["J_final"][b], result["final_x"][b] = J, x
                _failure(result, callback, message=(
                    f"Learning target {cfg['target']} not reached at Dn={Dn}, Ds={Ds}, "
                    f"network={net_id}, pattern={mu} within t={cfg['max_train_time']:g}; "
                    f"last instantaneous overlap={overlap:.6g}. No failed network was discarded."),
                    network_id=int(net_id), pattern_index=mu, steps=steps, overlap=overlap)
            _record_boundary(result, callback, b, mu, steps, overlap, J, x)
        result["training_complete"][b] = True
        result["recall"][b] = _numpy_recall(result, b)
        result["recall_complete"][b] = True
        _notify(callback, result, "network_completed", network_id=int(net_id), network_index=b)




### Optional compiled PyTorch/CUDA training

In [ ]:
_TORCH_CHUNK_CU = None


_TORCH_CHUNK_SOURCE = r"""
def integrate_chunk(x: Tensor, J: Tensor, xi: Tensor, eta: Tensor,
                    active: Tensor, neural_noise: Tensor, synaptic_noise: Tensor,
                    dt: float, beta: float, alpha: float, tauJ: float,
                    target: float, noise_x: float, noise_J: float,
                    start_step: int) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
    hit_steps = torch.full((x.size(0),), -1, dtype=torch.int64, device=x.device)
    hit_overlaps = torch.full_like(x[:, 0], float("nan"))
    for k in range(neural_noise.size(0)):
        row_norm = (J * J).sum(dim=2)
        x_new = x + dt * (-x + torch.tanh(beta * (torch.bmm(J, x.unsqueeze(2)).squeeze(2) + eta)))
        x_new = x_new + noise_x * neural_noise[k]
        J_new = J * (1.0 + (dt / tauJ) * alpha * (1.0 - row_norm)).unsqueeze(2)
        J_new = J_new + (dt / (tauJ * x.size(1))) * (xi - x).unsqueeze(2) * x.unsqueeze(1)
        J_new = J_new + noise_J * synaptic_noise[k]
        J_new.diagonal(0, 1, 2).zero_()
        J = torch.where(active[:, None, None], J_new, J)
        x = torch.where(active[:, None], x_new, x)
        overlap = (x * xi).mean(dim=1)
        done = active & (overlap >= target)
        hit_steps = torch.where(done, torch.full_like(hit_steps, start_step + k + 1), hit_steps)
        hit_overlaps = torch.where(done, overlap, hit_overlaps)
        active = active & ~done
    return x, J, active, hit_steps, hit_overlaps
"""


def _torch_chunk_integrator():
    # CompilationUnit works when this source is executed in a notebook cell;
    # unlike @torch.jit.script it does not require an inspectable .py file.
    global _TORCH_CHUNK_CU
    if _TORCH_CHUNK_CU is None:
        import torch
        _TORCH_CHUNK_CU = torch.jit.CompilationUnit(_TORCH_CHUNK_SOURCE)
    return _TORCH_CHUNK_CU.integrate_chunk


def _train_torch(result: dict[str, Any], callback: Callable | None,
                 device: str) -> None:
    """Batched Torch integration with exact first passage inside compiled chunks.

    Noise streams are independent for every network, memory and noise type.
    Unused samples in the last chunk never affect the next memory's stream.
    Frozen batch members receive no further state or synaptic updates.  The
    host sees one compact status transfer per chunk; thresholds are checked
    every dt on the device, and recorded learning times are the exact step
    of first passage, not the chunk boundary.  CUDA RNG sequences can vary
    with chunk size; the stochastic equation and switching rule do not.
    """
    import torch

    cfg, Dn, Ds = result["config"], result["Dn"], result["Ds"]
    B, N, P = len(result["net_ids"]), cfg["N"], cfg["P"]
    dtype = torch.float32 if cfg["dtype"] == "float32" else torch.float64
    dt, tauJ = cfg["dt"], cfg["tauJ"]
    max_steps = int(round(cfg["max_train_time"] / dt))
    noise_x, noise_J = math.sqrt(2 * Dn * dt), math.sqrt(2 * Ds * dt / N) / tauJ
    chunk_fn = _torch_chunk_integrator()
    J = torch.as_tensor(result["J_initial"], dtype=dtype, device=device).clone()
    x = torch.as_tensor(result["initial_x"], dtype=dtype, device=device).clone()
    xi_all = torch.as_tensor(result["xi"], dtype=dtype, device=device)
    eta_all = torch.as_tensor(result["eta"], dtype=dtype, device=device)
    reset_rng = [_rng(cfg["seed"], int(net_id), 6) for net_id in result["net_ids"]]
    old_tf32 = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False
    old_num_threads = torch.get_num_threads()
    if device == "cpu":
        torch.set_num_threads(cfg["blas_threads"])
    try:
        with torch.no_grad():
            for mu in range(P):
                # Pattern-separated streams make discarded, pre-generated
                # post-hit samples irrelevant to all subsequent patterns.
                gn = [torch.Generator(device=device).manual_seed(
                    _torch_seed(cfg["seed"], int(net_id), 3, mu)) for net_id in result["net_ids"]]
                gw = [torch.Generator(device=device).manual_seed(
                    _torch_seed(cfg["seed"], int(net_id), 4, mu)) for net_id in result["net_ids"]]
                if mu and cfg["reset_between_patterns"]:
                    reset = np.stack([_initial_x(rng, (N,), cfg["train_init"],
                                               np.dtype(cfg["dtype"])) for rng in reset_rng])
                    x = torch.as_tensor(reset, dtype=dtype, device=device)
                xi, eta = xi_all[:, mu], eta_all[:, mu]
                overlap_cpu = (x * xi).mean(dim=1).cpu().numpy()
                active_cpu = overlap_cpu < cfg["target"]
                active = torch.as_tensor(active_cpu, device=device)
                for b in np.flatnonzero(~active_cpu):
                    _record_boundary(result, callback, int(b), mu, 0, float(overlap_cpu[b]),
                                     J[b].cpu().numpy(), x[b].cpu().numpy())
                steps = 0
                while active_cpu.any() and steps < max_steps:
                    K = min(cfg["torch_chunk_size"], max_steps - steps)
                    neural_noise = torch.zeros((K, B, N), dtype=dtype, device=device)
                    synaptic_noise = torch.zeros((K, B, N, N), dtype=dtype, device=device)
                    for b in np.flatnonzero(active_cpu):
                        if Dn:
                            neural_noise[:, b] = torch.randn((K, N), generator=gn[b], device=device, dtype=dtype)
                        if Ds:
                            synaptic_noise[:, b] = torch.randn((K, N, N), generator=gw[b], device=device, dtype=dtype)
                    x, J, active, hit_steps, hit_overlaps = chunk_fn(
                        x, J, xi, eta, active, neural_noise, synaptic_noise,
                        dt, cfg["beta"], cfg["alpha"], tauJ, cfg["target"],
                        noise_x, noise_J, steps)
                    steps += K
                    # These transfers occur only every K steps.  Full arrays
                    # are copied only on first passage or a failure.
                    hit_cpu = hit_steps.cpu().numpy()
                    active_cpu = active.cpu().numpy()
                    overlap_cpu = (x * xi).mean(dim=1).cpu().numpy()
                    finite_J = torch.isfinite(J).all(dim=2).all(dim=1).cpu().numpy()
                    finite = np.isfinite(overlap_cpu) & finite_J
                    if not finite.all():
                        b = int(np.flatnonzero(~finite)[0])
                        result["J_final"][:] = J.cpu().numpy()
                        result["final_x"][:] = x.cpu().numpy()
                        _failure(result, callback, message=(
                            f"Nonfinite Torch state at Dn={Dn}, Ds={Ds}, network={result['net_ids'][b]}, "
                            f"pattern={mu}, by t={steps * dt:g}."), network_id=int(result["net_ids"][b]),
                            pattern_index=mu, steps=steps, overlap=float(overlap_cpu[b]))
                    if np.any(hit_cpu >= 0):
                        hit_overlap_cpu = hit_overlaps.cpu().numpy()
                        for b in np.flatnonzero(hit_cpu >= 0):
                            _record_boundary(result, callback, int(b), mu, int(hit_cpu[b]),
                                             float(hit_overlap_cpu[b]), J[b].cpu().numpy(), x[b].cpu().numpy())
                if active_cpu.any():
                    b = int(np.flatnonzero(active_cpu)[0])
                    result["J_final"][:] = J.cpu().numpy()
                    result["final_x"][:] = x.cpu().numpy()
                    _failure(result, callback, message=(
                        f"Learning target {cfg['target']} not reached at Dn={Dn}, Ds={Ds}, "
                        f"network={result['net_ids'][b]}, pattern={mu} within t={cfg['max_train_time']:g}; "
                        f"last instantaneous overlap={overlap_cpu[b]:.6g}. No failed network was discarded."),
                        network_id=int(result["net_ids"][b]), pattern_index=mu, steps=steps,
                        overlap=float(overlap_cpu[b]))
            result["training_complete"][:] = True
            R = cfg["n_recall"]
            recall_x = np.stack([_initial_x(_rng(cfg["seed"], int(net_id), 5),
                (P, R, N), cfg["recall_init"], np.dtype(cfg["dtype"])) for net_id in result["net_ids"]])
            xr = torch.as_tensor(recall_x.reshape(B, P * R, N), dtype=dtype, device=device)
            er = eta_all.repeat_interleave(R, dim=1)
            tr = xi_all.repeat_interleave(R, dim=1)
            n_steps = int(round(cfg["recall_time"] / dt))
            burn_step = int(round(cfg["recall_burn"] / dt))
            sums = torch.zeros((B, P * R), dtype=torch.float64, device=device)
            count = 0
            if burn_step == 0:
                sums += (xr * tr).mean(dim=2)
                count += 1
            for step in range(1, n_steps + 1):
                xr += dt * (-xr + torch.tanh(cfg["beta"] * (torch.bmm(xr, J.transpose(1, 2)) + er)))
                if step >= burn_step:
                    sums += (xr * tr).mean(dim=2)
                    count += 1
            recalls = (sums / count).reshape(B, P, R).cpu().numpy()
            if not np.isfinite(recalls).all():
                raise FloatingPointError("Torch recall integration produced nonfinite values.")
            result["recall"][:] = recalls
            result["recall_complete"][:] = True
            for b, net_id in enumerate(result["net_ids"]):
                _notify(callback, result, "network_completed", network_id=int(net_id), network_index=b)
    finally:
        torch.backends.cuda.matmul.allow_tf32 = old_tf32
        if device == "cpu":
            torch.set_num_threads(old_num_threads)




### Unified training entry point

In [ ]:
def train_condition(cfg: Mapping[str, Any], Dn: float, Ds: float,
                    net_ids: Sequence[int], checkpoint_callback: Callable | None = None
                    ) -> dict[str, Any]:
    """Train P independent I/O pairs sequentially and evaluate every memory.

    Required user-facing sizes: N=200, P=20.  Smaller values are supported
    only for explicit software checks.  ``device='auto'`` chooses CUDA if
    available and otherwise uses NumPy. ``device='cpu'`` or ``'numpy'``
    always chooses NumPy. ``device='torch_cpu'`` is an explicit Torch CPU
    verification backend. ``device='cuda'``/``'cuda:0'`` requires PyTorch
    and a usable GPU, and raises if unavailable instead of silently
    changing the requested device.

    All arrays are NumPy arrays on return.  ``s_boundary[b, ell, mu, i]``
    stores stability of target mu immediately after training target ell.
    ``q[b, mu, i] = xi[b, mu, i] * eta[b, mu, i]``.  Training times are
    ``learn_steps * dt``.  Recall output axes are (network, memory, initial
    state); ``recall.mean(axis=(1, 2))`` is each network's capacity.

    ``TrainingFailure.partial_result`` and the optional callback preserve
    interrupted states.  Completed networks are never selectively removed.
    """
    cfg = _sim_config(cfg)
    for name, value in (("Dn", Dn), ("Ds", Ds)):
        if not np.isfinite(value) or value < 0:
            raise ValueError(f"{name} must be finite and nonnegative.")
    ids_list = list(net_ids)
    if not ids_list or any(isinstance(i, bool) or not isinstance(i, (int, np.integer)) or i < 0 for i in ids_list):
        raise ValueError("net_ids must be a nonempty sequence of nonnegative integers.")
    ids = np.asarray(ids_list, dtype=np.int64)
    if len(np.unique(ids)) != len(ids):
        raise ValueError("net_ids must be unique.")
    device = str(cfg["device"]).lower()
    if device in ("cpu", "numpy"):
        backend = "numpy"
    elif device == "torch_cpu":
        try:
            import torch
        except ImportError as error:
            raise RuntimeError("device='torch_cpu' requires an installed PyTorch.") from error
        backend = "torch_cpu"
        device = "cpu"
    elif device == "auto" or device.startswith("cuda"):
        try:
            import torch
            cuda_available = torch.cuda.is_available()
        except ImportError:
            cuda_available = False
        if not cuda_available and device != "auto":
            raise RuntimeError("Explicit CUDA device requested, but PyTorch/CUDA is unavailable.")
        backend = "torch_cuda" if cuda_available else "numpy"
        if device == "auto":
            device = "cuda" if cuda_available else "numpy"
    else:
        raise ValueError("device must be 'auto', 'numpy', 'cpu', 'torch_cpu', 'cuda' or e.g. 'cuda:0'.")
    result = _make_result(cfg, float(Dn), float(Ds), ids, backend)
    start = time.perf_counter()
    try:
        if backend == "numpy":
            try:
                from threadpoolctl import threadpool_limits
                context = threadpool_limits(limits=cfg["blas_threads"])
            except ImportError:
                context = nullcontext()
            with context:
                _train_numpy(result, checkpoint_callback)
        else:
            _train_torch(result, checkpoint_callback, device)
    finally:
        result["wall_seconds"] = time.perf_counter() - start
    if not result["training_complete"].all() or not result["recall_complete"].all():
        raise RuntimeError("Internal completion invariant failed.")
    result["learn_time"] = result["learn_steps"] * cfg["dt"]
    result["s_final"] = result["s_boundary"][:, -1].copy()
    result["s_birth"] = result["s_boundary"][:, np.arange(cfg["P"]), np.arange(cfg["P"])].copy()
    result["capacity_per_network"] = result["recall"].mean(axis=(1, 2))
    _notify(checkpoint_callback, result, "condition_completed")
    return result



## 3. Resumable simulation and compact checkpoints

Only the data needed for Fig. 3 are saved for each finished network (`recall`, `learn_steps`, `learn_overlap` plus identifying metadata); the much larger intermediate matrices and stability histories used by the AMF analysis are **not** saved. This does not affect the original training or recall computations.

A run identity records the simulation parameters, grid, seed, cohort sizes, backend, and software version so differently configured runs cannot silently share checkpoints. Data files are written atomically and loaded with `allow_pickle=False`. A failed or incomplete network is never silently discarded.

In [ ]:
import hashlib
import json
import os
import platform
import sys
import time

WORKFLOW_VERSION = "fig3-abc-from-original-simulator-v1.0"

def noise_key(dn, ds):
    return (round(float(dn), 8), round(float(ds), 8))

def condition_label(key):
    return f"Dn_{key[0]:.5f}_Ds_{key[1]:.5f}".replace(".", "p")

def all_noise_conditions():
    keys = {noise_key(dn, ds) for ds in (0.01, 1.0) for dn in BASIC_GRID}
    keys.update(noise_key(dn, ds) for dn in (0.01, 1.0) for ds in BASIC_GRID)
    keys.update(noise_key(dn, 0.01) for dn in FIG3A_DN)
    return sorted(keys)

CONDITIONS = all_noise_conditions()
FIG3A_KEYS = {noise_key(dn, 0.01) for dn in FIG3A_DN}
EXPECTED_COUNTS = {
    key: N_FIG3A_NETWORKS if key in FIG3A_KEYS else N_NETWORKS
    for key in CONDITIONS
}
print(f"{len(CONDITIONS)} unique noise conditions; "
      f"{sum(EXPECTED_COUNTS.values())} trained networks")
if RUN_MODE == "paper":
    assert len(CONDITIONS) == 80 and sum(EXPECTED_COUNTS.values()) == 880

SIM_CFG = _sim_config(SIM_SETTINGS)

def _backend_description(device):
    try:
        import torch
        cuda = torch.cuda.is_available()
        torch_version = str(torch.__version__)
        gpu = torch.cuda.get_device_name(0) if cuda else None
    except ImportError:
        cuda, torch_version, gpu = False, None, None
    if device == "auto":
        name = "torch_cuda" if cuda else "numpy"
    elif device in ("numpy", "cpu"):
        name = "numpy"
    elif device == "torch_cpu":
        name = "torch_cpu"
    elif device.startswith("cuda"):
        if not cuda:
            raise RuntimeError("CUDA was requested but is not available")
        name = "torch_cuda"
    else:
        raise ValueError(f"Unsupported device: {device}")
    return dict(backend=name, torch_version=torch_version, gpu=gpu)

BACKEND = _backend_description(SIM_CFG["device"])
IDENTITY = {
    "version": WORKFLOW_VERSION,
    "run_mode": RUN_MODE,
    "simulation_config": SIM_CFG,
    "backend": BACKEND,
    "basic_grid": BASIC_GRID,
    "fig3a_dn": FIG3A_DN,
    "normal_networks": N_NETWORKS,
    "fig3a_networks": N_FIG3A_NETWORKS,
    "batch_size": BATCH_SIZE,
    "conditions": [list(k) for k in CONDITIONS],
}
RUN_HASH = hashlib.sha256(json.dumps(IDENTITY, sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR = OUTPUT_BASE / f"run_{RUN_HASH}"
for folder in ("raw", "tables", "figures"):
    (RUN_DIR / folder).mkdir(parents=True, exist_ok=True)

MANIFEST = RUN_DIR / "run_config.json"
if MANIFEST.exists():
    previous = json.loads(MANIFEST.read_text(encoding="utf-8"))
    if previous["identity"] != IDENTITY:
        raise RuntimeError(f"Run identity mismatch: {MANIFEST}")
else:
    metadata = dict(identity=IDENTITY, created_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                    environment=dict(python=sys.version, numpy=np.__version__,
                                     platform=platform.platform()),
                    comment="Source: authors' Fig. 3 simulation; compact recall-only checkpoints")
    temporary = MANIFEST.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
    os.replace(temporary, MANIFEST)

print("Checkpoint directory:", RUN_DIR)
print("Resolved backend:", BACKEND["backend"])


In [ ]:
def checkpoint_file(key, network_id):
    return RUN_DIR / "raw" / condition_label(key) / f"network_{network_id:06d}.npz"

def save_checkpoint(key, network_id, batch, index):
    path = checkpoint_file(key, network_id)
    path.parent.mkdir(parents=True, exist_ok=True)
    meta = dict(run_hash=RUN_HASH, Dn=key[0], Ds=key[1], network_id=int(network_id))
    tmp = path.with_suffix(".npz.part")
    with open(tmp, "wb") as f:
        np.savez_compressed(
            f, recall=np.asarray(batch["recall"][index], dtype=np.float64),
            learn_steps=np.asarray(batch["learn_steps"][index], dtype=np.int64),
            learn_overlap=np.asarray(batch["learn_overlap"][index], dtype=np.float64),
            metadata=np.asarray(json.dumps(meta, sort_keys=True)),
        )
    os.replace(tmp, path)

def load_checkpoint(key, network_id):
    path = checkpoint_file(key, network_id)
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing Fig.3 training checkpoint {path}; run training first."
        )
    with np.load(path, allow_pickle=False) as data:
        meta = json.loads(str(data["metadata"].item()))
        recall = np.asarray(data["recall"], dtype=float)
        steps = np.asarray(data["learn_steps"], dtype=np.int64)
        overlap = np.asarray(data["learn_overlap"], dtype=float)
    if (meta != dict(run_hash=RUN_HASH, Dn=key[0], Ds=key[1], network_id=int(network_id))
        or recall.shape != (SIM_CFG["P"], SIM_CFG["n_recall"])
        or steps.shape != (SIM_CFG["P"],)
        or overlap.shape != (SIM_CFG["P"],)
        or np.any(steps < 0)
        or not np.isfinite(recall).all()
        or not np.isfinite(overlap).all()):
        raise RuntimeError(f"Invalid or incomplete saved network: {path}")
    return recall, steps, overlap

def run_full_training():
    total = len(CONDITIONS)
    for idx, key in enumerate(CONDITIONS, start=1):
        n = EXPECTED_COUNTS[key]
        missing = []
        for network_id in range(n):
            file = checkpoint_file(key, network_id)
            if file.exists():
                load_checkpoint(key, network_id)  # verify rather than silently reuse corruption
            else:
                missing.append(network_id)
        print(f"[{idx:02}/{total}] Dn={key[0]:g}, Ds={key[1]:g}: "
              f"{n-len(missing)}/{n} complete", flush=True)
        for k in range(0, len(missing), BATCH_SIZE):
            network_ids = missing[k:k+BATCH_SIZE]
            batch = train_condition(SIM_CFG, key[0], key[1], network_ids)
            for pos, network_id in enumerate(network_ids):
                save_checkpoint(key, network_id, batch, pos)
            print("   saved networks", network_ids, flush=True)
    print("All conditions trained successfully.")

if RUN_TRAINING:
    run_full_training()
else:
    print("Training disabled: checking and plotting already-completed checkpoints only.")


## 4. Aggregate recalls and compute independent-network SEM

For each trained network, we first average the five recall initial states to obtain the mean recall for each stored memory. Fig. 3A then averages over the **30 network-level means**. The capacity for one network is the mean over all 20 memories and five initial states. Fig. 3B/C then average that quantity across the independently trained networks at each noise condition, reporting SEM with denominator `sqrt(n_networks)`.

For auditing and secondary plotting, export individual recall trials and their network-level / condition-level aggregates as CSV. No new simulations are performed in this step.

In [ ]:
import pandas as pd

def mean_sem(values):
    x = np.asarray(values, dtype=float)
    if not (x.ndim == 1 and len(x) >= 2 and np.isfinite(x).all()):
        raise ValueError("SEM requires at least two finite independent networks")
    return float(np.mean(x)), float(np.std(x, ddof=1) / np.sqrt(len(x)))

def make_figure_data():
    trial_rows, memory_rows, capacity_rows = [], [], []
    for key in CONDITIONS:
        for network_id in range(EXPECTED_COUNTS[key]):
            recall, steps, overlap = load_checkpoint(key, network_id)
            for memory in range(SIM_CFG["P"]):
                memory_rows.append(dict(Dn=key[0], Ds=key[1], network=network_id,
                                        memory=memory, overlap=float(recall[memory].mean())))
                for start, value in enumerate(recall[memory]):
                    trial_rows.append(dict(Dn=key[0], Ds=key[1], network=network_id,
                                           memory=memory, recall_start=start, overlap=float(value)))
            capacity_rows.append(dict(Dn=key[0], Ds=key[1], network=network_id,
                                      capacity=float(recall.mean()),
                                      training_steps_total=int(steps.sum())))
    trials = pd.DataFrame(trial_rows)
    netmem = pd.DataFrame(memory_rows)
    netcap = pd.DataFrame(capacity_rows)

    # .agg() computes the mean / sample standard deviation over trained networks.
    summary_a = (netmem[netmem["Ds"].eq(0.01) & netmem["Dn"].isin(FIG3A_DN)]
                 .groupby(["Dn", "Ds", "memory"])["overlap"]
                 .agg(mean="mean", std="std", n_networks="count").reset_index())
    summary_a["sem"] = summary_a["std"] / np.sqrt(summary_a["n_networks"])
    summary_bc = (netcap.groupby(["Dn", "Ds"])["capacity"]
                  .agg(mean="mean", std="std", n_networks="count").reset_index())
    summary_bc["sem"] = summary_bc["std"] / np.sqrt(summary_bc["n_networks"])

    assert len(netcap) == sum(EXPECTED_COUNTS.values())
    assert len(summary_bc) == len(CONDITIONS)
    assert len(summary_a) == len(FIG3A_DN) * SIM_CFG["P"]
    for key in CONDITIONS:
        this = summary_bc[(summary_bc["Dn"] == key[0]) & (summary_bc["Ds"] == key[1])]
        assert len(this) == 1 and int(this["n_networks"].iloc[0]) == EXPECTED_COUNTS[key]
        assert np.isfinite(this[["mean", "sem"]].values).all()
    for dn in FIG3A_DN:
        assert (summary_a[summary_a["Dn"] == dn]["n_networks"] == N_FIG3A_NETWORKS).all()

    tables = RUN_DIR / "tables"
    for name, dataframe in (
        ("fig3_recall_trials.csv", trials),
        ("fig3_network_memory.csv", netmem),
        ("fig3_network_capacity.csv", netcap),
        ("fig3A_summary.csv", summary_a),
        ("fig3BC_summary.csv", summary_bc),
    ):
        dataframe.to_csv(tables / name, index=False, float_format="%.16g")
    print("Saved Fig. 3 data tables:", tables)
    return summary_a, summary_bc

FIG3A_DATA, FIG3BC_DATA = make_figure_data()
display(FIG3BC_DATA.head()) if "display" in globals() else print(FIG3BC_DATA.head())


## 5. Draw and export Fig. 3A–C

Plots are generated from the network-aggregated CSV tables. Each point's SEM is across **independently trained networks**; Fig. 3A memories are ordered oldest → newest. Individual panels and a combined 3-panel figure are exported in vector PDF and high-resolution PNG. Values are not rescaled or smoothed.

In [ ]:
plt.rcParams.update({
    "font.size": 10.5, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

def draw_fig3A(ax, data):
    for dn in FIG3A_DN:
        z = data[data["Dn"] == dn].sort_values("memory")
        ax.errorbar(z["memory"], z["mean"], yerr=z["sem"],
                    marker="o", markersize=3, linewidth=1.3, capsize=2,
                    label=fr"$D_n={dn:g}$")
    if SIM_CFG["P"] >= 20:
        ax.set_xticks([0, 5, 10, 15, 19])
    else:
        ax.set_xticks(range(SIM_CFG["P"]))
    ax.set_xlabel("Memory index (earlier → later)")
    ax.set_ylabel("Recall overlap")
    ax.set_ylim(0, 1.05)
    ax.set_title(fr"A   $D_s=0.01$", loc="left")
    ax.legend(fontsize=8, frameon=False)

def draw_fig3B(ax, data):
    for ds in (0.01, 1.0):
        z = data[data["Ds"] == ds].sort_values("Dn")
        ax.errorbar(z["Dn"], z["mean"], yerr=z["sem"],
                    marker="o", markersize=3, linewidth=1.3, capsize=2,
                    label=fr"$D_s={ds:g}$")
    ax.set_xlabel(r"Neural noise intensity $D_n$")
    ax.set_ylabel("Capacity (mean recall overlap)")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.set_title("B", loc="left")
    ax.legend(fontsize=9, frameon=False)

def draw_fig3C(ax, data):
    for dn in (0.01, 1.0):
        z = data[data["Dn"] == dn].sort_values("Ds")
        ax.errorbar(z["Ds"], z["mean"], yerr=z["sem"],
                    marker="o", markersize=3, linewidth=1.3, capsize=2,
                    label=fr"$D_n={dn:g}$")
    ax.set_xlabel(r"Synaptic noise intensity $D_s$")
    ax.set_ylabel("Capacity (mean recall overlap)")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.set_title("C", loc="left")
    ax.legend(fontsize=9, frameon=False)

def save_figure(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(RUN_DIR / "figures" / f"{name}.{ext}",
                    dpi=250, bbox_inches="tight")

def create_figures():
    tasks = (
        ("Fig3A", draw_fig3A, FIG3A_DATA),
        ("Fig3B", draw_fig3B, FIG3BC_DATA),
        ("Fig3C", draw_fig3C, FIG3BC_DATA),
    )
    for name, draw, data in tasks:
        fig, ax = plt.subplots(figsize=(6.2, 3.9))
        draw(ax, data)
        fig.tight_layout()
        save_figure(fig, name)
        if SHOW_FIGURES:
            plt.show()
        plt.close(fig)
    fig, axes = plt.subplots(1, 3, figsize=(16.2, 4.4))
    for ax, (_, draw, data) in zip(axes, tasks):
        draw(ax, data)
    fig.tight_layout(w_pad=2.2)
    save_figure(fig, "Fig3_ABC")
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)
    print("Saved Fig. 3A/B/C and combined figure:", RUN_DIR / "figures")

create_figures()


## 6. How to rerun and verify

- **Run everything from scratch**: use `RUN_MODE="paper"`, `RUN_TRAINING=True`, and execute cells from the top. Completed network checkpoints are saved after each batch and skipped when rerunning with the same parameters.
- **Redraw existing results without retraining**: keep `RUN_MODE` and all numerical/backend settings unchanged, set `RUN_TRAINING=False`, then run all cells. Missing checkpoints raise an error rather than silently generating partial figures.
- **Quick software test**: set `RUN_MODE="smoke"` and run all cells. This changes `N`, `P`, numerical times, sample counts and the noise grid, so the outputs must **not** be presented as the paper figures. Smoke mode writes to a distinct fingerprinted folder.
- **Numerical reproducibility**: The seed is `1729`, with stable network-specific RNG streams. A GPU run can differ at the realization level from a CPU/NumPy run because the random generators and floating-point backends differ. Results are never silently pooled across different backends or numerical parameters.
- **Paper provenance**: This notebook uses the original full-simulation routine, but the full 880-network paper run has **not** been executed as part of producing this clean public notebook. Numerical agreement with the published plots requires carrying out that run (or loading its completed checkpoints), not just opening the notebook.

**Suggested citation:** cite the Kurikawa–Kaneko article when using this notebook; check the published version for complete journal/DOI metadata.